In [1]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

modelo = tf.keras.models.load_model('/home/maria-luna/mejor_modelo.keras')
clases = ['capacitor', 'diodo', 'led', 'resistencia', 'transistor']

# Descripción de cada componente
descripciones = {
    'capacitor':   'Almacena y libera energía eléctrica. Se usa en filtros, fuentes de poder y circuitos de temporización.',
    'diodo':       'Permite el flujo de corriente en una sola dirección. Usado en rectificadores y protección de circuitos.',
    'led':         'Diodo emisor de luz. Convierte electricidad en luz. Muy eficiente energéticamente.',
    'resistencia': 'Limita el flujo de corriente eléctrica. Componente fundamental en cualquier circuito electrónico.',
    'transistor':  'Amplifica o conmuta señales eléctricas. Es la base de todos los circuitos digitales modernos.'
}

colores = {
    'capacitor':   '#3B82F6',
    'diodo':       '#8B5CF6',
    'led':         '#F59E0B',
    'resistencia': '#10B981',
    'transistor':  '#EF4444'
}

print("✓ Modelo cargado")

/home/maria-luna/proyectos/componentes-ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1778960390.434876    6040 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778960391.133176    6040 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778960393.643935    6040 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1778960396.078581    6040 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


✓ Modelo cargado


In [2]:
def predecir(imagen):
    if imagen is None:
        return "⚠️ Sube una imagen para identificar el componente."
    
    img = Image.fromarray(imagen).convert('RGB').resize((224, 224))
    arr = np.array(img, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)
    
    pred = modelo.predict(arr, verbose=0)[0]
    idx = np.argmax(pred)
    clase = clases[idx]
    confianza = pred[idx] * 100
    
    # Construir resultado HTML
    color = colores[clase]
    desc = descripciones[clase]
    
    barras = ""
    for i, c in enumerate(clases):
        pct = pred[i] * 100
        barras += f"""
        <div style='margin:6px 0'>
            <div style='display:flex; justify-content:space-between; font-size:13px'>
                <span style='font-weight:{"700" if i==idx else "400"}'>{c}</span>
                <span>{pct:.1f}%</span>
            </div>
            <div style='background:#e5e7eb; border-radius:4px; height:10px'>
                <div style='background:{"" if i!=idx else color}; background:{color if i==idx else "#9CA3AF"}; 
                     width:{pct}%; height:10px; border-radius:4px'></div>
            </div>
        </div>"""
    
    html = f"""
    <div style='font-family:sans-serif; padding:16px'>
        <div style='background:{color}22; border-left:5px solid {color}; 
             padding:16px; border-radius:8px; margin-bottom:16px'>
            <h2 style='margin:0; color:{color}; font-size:24px'>
                {clase.upper()}
            </h2>
            <p style='margin:8px 0 0; color:#374151; font-size:14px'>{desc}</p>
            <p style='margin:8px 0 0; color:#6B7280; font-size:13px'>
                Confianza: <strong>{confianza:.1f}%</strong>
            </p>
        </div>
        <h4 style='color:#374151; margin-bottom:8px'>Probabilidades:</h4>
        {barras}
    </div>"""
    
    return html

demo = gr.Interface(
    fn=predecir,
    inputs=gr.Image(label="📷 Sube una foto del componente"),
    outputs=gr.HTML(label="Resultado"),
    title="🔍 Reconocedor de Componentes Electrónicos",
    description="Sube una imagen de un componente electrónico y el modelo lo identificará automáticamente usando Inteligencia Artificial.",
    examples=[
        ['/home/maria-luna/dataset/test/led/000059.jpg'],
        ['/home/maria-luna/dataset/test/resistencia/000001.jpg'],
        ['/home/maria-luna/dataset/test/transistor/000001.jpg'],
        ['/home/maria-luna/dataset/test/capacitor/000001.jpg'],
        ['/home/maria-luna/dataset/test/diodo/000001.jpg'],
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=False)

/home/maria-luna/proyectos/componentes-ml/lib/python3.13/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
